# FlashNystrom — Colab experiments → paper artifacts

Runs the paper experiments and emits **PDF figures + JSON** you can drop straight into the manuscript. One run → `flashnystrom_artifacts.zip`.

**GPU:** kernels are **sm_80+** — use **A100 or L4** (Colab Pro/Pro+). A **T4 (sm_75) will NOT work.** *Runtime → Change runtime type → A100.*

**Design:** accuracy runs use the **validated fixed-batch recipe** (grad_clip, batch 256/128) so recall is sound; auto-batch is used only by the throughput profiler (where saturating the GPU is the point).

## 0. Check the GPU

In [ ]:
import torch
name = torch.cuda.get_device_name(); cap = torch.cuda.get_device_capability()
print(name, '| compute capability', cap)
assert cap[0] >= 8, f'Need sm_80+ (A100/L4); this is {name} sm_{cap[0]}{cap[1]}. Switch runtime.'
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## 1. Get the code
Push your repo first (`paper/*.py` is tracked, the manuscript is gitignored), then set `REPO_URL`.

In [ ]:
REPO_URL = 'https://github.com/athrva98/FlashNystrom.git'
!git clone $REPO_URL flashnystrom
%cd flashnystrom
# alt: from google.colab import drive; drive.mount('/content/drive')  # then unzip your repo

## 2. Build the fused CUDA kernels (~3–6 min)

In [ ]:
import os, torch
cap = torch.cuda.get_device_capability()
os.environ['TORCH_CUDA_ARCH_LIST'] = f'{cap[0]}.{cap[1]}'
!pip install -e . --no-build-isolation

## 3. Verify the kernels (forward matches reference, backward finite)

In [ ]:
import torch
from flash_nystrom import flash_nystrom_attention
from flash_nystrom.reference import nystrom_attention_reference
mk = lambda: torch.randn(4, 2, 256, 64, device='cuda', dtype=torch.bfloat16)
q, k, v = mk(), mk(), mk()
o = flash_nystrom_attention(q, k, v, 64, 6); r = nystrom_attention_reference(q, k, v, 64, 6)
print('fwd finite:', bool(torch.isfinite(o).all()), ' max|fn-ref|:', (o.float()-r.float()).abs().max().item())
qg = q.clone().requires_grad_(True); flash_nystrom_attention(qg, k, v, 64, 6).sum().backward()
print('bwd grad finite:', bool(torch.isfinite(qg.grad).all()))

## 4. Scaling / crossover  → `scaling.json`
Throughput + peak memory vs N at the **auto-found max batch** (saturates the GPU). Subprocess-isolated, so sdpa OOMing at large N doesn't kill the sweep. (~10–20 min)

In [ ]:
!python benchmarks/profile_scaling.py \
  --backends sdpa flash_nystrom nystrom_reference \
  --Ns 256 512 1024 2048 4096 8192 16384 \
  --json scaling.json

## 5. MQAR recall  → `mqar_length.json`, `mqar_capacity.json`
**Length sweep** = recall vs context (and the faithfulness check: flash_nystrom ≈ nystrom_reference ≈ sdpa). **Capacity** = the honest rank-limit ablation. Fixed-batch recipe + grad_clip, internal best-over-LR. (longest step — trim `--seq_lens`/`--lrs` to iterate)

In [ ]:
!python -m paper.mqar.run_scaling_sweep --mode length \
  --backends sdpa flash_nystrom nystrom_reference \
  --seq_lens 256 512 1024 2048 4096 --num_kv_pairs 16 \
  --json mqar_length.json

In [ ]:
!python -m paper.mqar.run_scaling_sweep --mode capacity \
  --backends sdpa flash_nystrom nystrom_reference \
  --seq_len 1024 --kv_pairs 16 32 64 128 256 \
  --json mqar_capacity.json

## 6. CIFAR pixel-token (long-context vision)  → `three_way_results.json`
`patch_size=1` → 1025-token sequence, from scratch, fixed batch + grad_clip. (~30–60 min)

In [ ]:
!python benchmarks/train_three_way.py \
  --patch_size 1 --epochs 30 --grad_clip 1.0 \
  --backends sdpa nystrom_reference flash_nystrom

## 7. Build figures + download artifacts
Generates the paper PDFs from whatever JSON exists, then zips figures + JSON.

In [ ]:
!python benchmarks/make_figures.py
import glob, zipfile
zf = 'flashnystrom_artifacts.zip'
paths = sorted(glob.glob('figures/*.pdf') + glob.glob('*.json'))
with zipfile.ZipFile(zf, 'w') as z:
    for p in paths: z.write(p)
print('packed', len(paths), 'files into', zf); print('\n'.join(paths))
try:
    from google.colab import files; files.download(zf)
except Exception:
    print('(not on Colab) artifacts at', zf)

## Optional — 3-seed error bars on MQAR recall
The figures above use one seed (best-over-LR). For mean±std on the headline recall (heads=2), run this — slower. Numbers print to stdout.

In [ ]:
!python -m paper.mqar.sweep \
  --backends sdpa flash_nystrom nystrom_reference \
  --heads 2 --inits normal --seeds 0 1 2 \
  --lrs 1e-3 3.16e-3 1e-2 3.16e-2 --grad_clip 1.0 --epochs 64

## Notes
- Run **one section per session** to economize CUs; section 5 is the longest.
- All accuracy numbers use grad_clip at the validated batch — that's the sound recall. Auto-batch is throughput-only (section 4).
- `display()` the PDFs inline with `from IPython.display import IFrame; IFrame('figures/scaling.pdf', 900, 400)`.